# Assignment 3 — IoT Sensor Data Analysis

This notebook analyzes the IoT sensor dataset and addresses all 26 questions from Parts A–E of the assignment.


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

file_path = 'iot_sensor_data_raw(1).csv'
df = pd.read_csv(file_path)

# Assignment-friendly aliases for the source dataset columns.
df['Device'] = df['Device_ID']
df['Battery'] = df['Battery_Level']
df['Factory'] = df['Location']
df.head()

## Part A — Data Preparation


### 1. Load the dataset

In [ ]:
df = pd.read_csv(file_path)
df['Device'] = df['Device_ID']
df['Battery'] = df['Battery_Level']
df['Factory'] = df['Location']
print('Dataset loaded successfully.')
df.head()

### 2. Display dimensions and structure

In [ ]:
print('Dimensions:', df.shape)
print('\nData types and structure:')
df.info()

print('\nColumns:')
print(df.columns.tolist())

### 3. Convert Timestamp into Pandas datetime

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
print(df['Timestamp'].dtype)
df.head()

### 4. Check whether timestamps are correctly ordered

In [ ]:
timestamps_ordered = df['Timestamp'].is_monotonic_increasing
print('Timestamps correctly ordered:', timestamps_ordered)

if not timestamps_ordered:
    print('\nNumber of out-of-order timestamp transitions:', (df['Timestamp'].diff() < pd.Timedelta(0)).sum())

### 5. Identify missing sensor readings

In [ ]:
sensor_columns = ['Temperature', 'Vibration', 'Battery_Level']
missing_counts = df.isna().sum()
print(missing_counts)

print('\nMissing sensor readings:')
display(df[sensor_columns].isna().sum().to_frame('Missing_Count'))

### 6. Percentage of missing values for each sensor

In [ ]:
missing_percentage = (df[sensor_columns].isna().mean() * 100).round(2)
missing_percentage.index = ['Temperature', 'Vibration', 'Battery']
missing_percentage = missing_percentage.to_frame('Missing_Percentage')
display(missing_percentage)

## Part B — Data Cleaning

For missing numerical sensor values, the notebook uses interpolation followed by forward/backward filling. This preserves the time-series nature of the sensor data while avoiding unnecessary row deletion.

### 7. Handle missing sensor values

In [ ]:
df_clean = df.copy()
df_clean['Device'] = df_clean['Device_ID']
df_clean['Battery'] = df_clean['Battery_Level']
df_clean['Factory'] = df_clean['Location']
df_clean = df_clean.sort_values('Timestamp').reset_index(drop=True)
for col in ['Temperature', 'Vibration', 'Battery']:
    df_clean[col] = df_clean[col].interpolate(method='linear', limit_direction='both')

print('Missing values after cleaning:')
display(df_clean[['Temperature', 'Vibration', 'Battery']].isna().sum().to_frame('Remaining_Missing'))

### Abnormal-reading thresholds

The assignment specifies battery thresholds explicitly. For temperature and vibration, thresholds are defined from the observed dataset using a robust IQR-based rule: values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are treated as abnormal. This avoids assuming engineering limits that are not provided in the assignment.

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

temp_low, temp_high = iqr_bounds(df_clean['Temperature'])
vib_low, vib_high = iqr_bounds(df_clean['Vibration'])

print(f'Temperature abnormal bounds: {temp_low:.2f} to {temp_high:.2f}')
print(f'Vibration abnormal bounds: {vib_low:.2f} to {vib_high:.2f}')

### 8. Identify abnormal temperature readings

In [ ]:
abnormal_temp = df_clean[(df_clean['Temperature'] < temp_low) | (df_clean['Temperature'] > temp_high)].copy()
print('Number of abnormal temperature readings:', len(abnormal_temp))
display(abnormal_temp)

### 9. Identify abnormal vibration readings

In [ ]:
abnormal_vibration = df_clean[(df_clean['Vibration'] < vib_low) | (df_clean['Vibration'] > vib_high)].copy()
print('Number of abnormal vibration readings:', len(abnormal_vibration))
display(abnormal_vibration)

### 10. Identify machines with critically low battery levels

In [ ]:
critical_battery = df_clean[df_clean['Battery'] < 20].copy()
print('Number of critically low battery readings:', len(critical_battery))
display(critical_battery)

### 11. How abnormal readings are defined

- **Battery:** the assignment explicitly defines `< 20` as Critical, `20–49` as Moderate, and `>= 50` as Healthy.
- **Temperature:** abnormal means outside the dataset's IQR-based bounds.
- **Vibration:** abnormal means outside the dataset's IQR-based bounds.
- The IQR rule is suitable here because the assignment does not provide fixed engineering thresholds for temperature or vibration.

## Part C — Time-Series Analysis


### 12. Average temperature by hour

In [ ]:
df_clean['Hour'] = df_clean['Timestamp'].dt.hour
avg_temperature_by_hour = df_clean.groupby('Hour')['Temperature'].mean().round(2)
display(avg_temperature_by_hour.to_frame('Average_Temperature'))

### 13. Average temperature for each device

In [ ]:
avg_temp_device = df_clean.groupby('Device')['Temperature'].mean().sort_values(ascending=False).round(2)
display(avg_temp_device.to_frame('Average_Temperature'))

### 14. Average vibration for each device

In [ ]:
avg_vibration_device = df_clean.groupby('Device')['Vibration'].mean().sort_values(ascending=False).round(2)
display(avg_vibration_device.to_frame('Average_Vibration'))

### 15. Maximum temperature recorded by each device

In [ ]:
max_temp_device = df_clean.groupby('Device')['Temperature'].max().sort_values(ascending=False).round(2)
display(max_temp_device.to_frame('Maximum_Temperature'))

### 16. Minimum battery level for each device

In [ ]:
min_battery_device = df_clean.groupby('Device')['Battery'].min().sort_values().round(2)
display(min_battery_device.to_frame('Minimum_Battery'))

### 17. Device with the highest average vibration

In [ ]:
highest_vibration_device = avg_vibration_device.idxmax()
highest_vibration_value = avg_vibration_device.max()
print(f'Device with highest average vibration: {highest_vibration_device}')
print(f'Average vibration: {highest_vibration_value:.2f}')

### 18. Factory with the highest average temperature

In [ ]:
avg_temp_factory = df_clean.groupby('Factory')['Temperature'].mean().sort_values(ascending=False).round(2)
display(avg_temp_factory.to_frame('Average_Temperature'))
highest_temp_factory = avg_temp_factory.idxmax()
print(f'Factory with highest average temperature: {highest_temp_factory}')

## Part D — Create New Variables


### 19. Battery_Status

In [ ]:
df_clean['Battery_Status'] = np.select(
    [df_clean['Battery'] >= 50, df_clean['Battery'] >= 20],
    ['Healthy', 'Moderate'],
    default='Critical'
)
display(df_clean[['Battery', 'Battery_Status']].head(10))

### 20. Temperature_Status

For consistency with the abnormal-temperature rule, values within the IQR bounds are Normal and values outside them are Abnormal.

In [ ]:
df_clean['Temperature_Status'] = np.where(
    df_clean['Temperature'].between(temp_low, temp_high),
    'Normal', 'Abnormal'
)
display(df_clean[['Temperature', 'Temperature_Status']].head(10))

### 21. Vibration_Status

In [ ]:
df_clean['Vibration_Status'] = np.where(
    df_clean['Vibration'].between(vib_low, vib_high),
    'Normal', 'Abnormal'
)
display(df_clean[['Vibration', 'Vibration_Status']].head(10))

### 22. Overall Machine_Health

A row is classified as:
- **Critical** if battery is Critical or either temperature/vibration is abnormal.
- **Warning** if battery is Moderate.
- **Normal** otherwise.

In [ ]:
critical_condition = (
    (df_clean['Battery_Status'] == 'Critical') |
    (df_clean['Temperature_Status'] == 'Abnormal') |
    (df_clean['Vibration_Status'] == 'Abnormal')
)
warning_condition = df_clean['Battery_Status'] == 'Moderate'

df_clean['Machine_Health'] = np.select(
    [critical_condition, warning_condition],
    ['Critical', 'Warning'],
    default='Normal'
)

display(df_clean[['Device', 'Battery_Status', 'Temperature_Status', 'Vibration_Status', 'Machine_Health']].head(20))
print('\nMachine health distribution:')
display(df_clean['Machine_Health'].value_counts().to_frame('Count'))

## Part E — Advanced Analysis


### 23. Devices that experienced critical conditions more than five times

A critical condition is counted whenever `Machine_Health == 'Critical'`.

In [ ]:
critical_counts = (
    df_clean[df_clean['Machine_Health'] == 'Critical']
    .groupby('Device')
    .size()
    .sort_values(ascending=False)
)
devices_over_five_critical = critical_counts[critical_counts > 5]
display(devices_over_five_critical.to_frame('Critical_Count'))

### 24. Factory with the highest number of abnormal sensor readings

In [ ]:
df_clean['Abnormal_Temperature'] = ~df_clean['Temperature'].between(temp_low, temp_high)
df_clean['Abnormal_Vibration'] = ~df_clean['Vibration'].between(vib_low, vib_high)
df_clean['Critical_Battery'] = df_clean['Battery'] < 20

df_clean['Abnormal_Sensor_Count'] = (
    df_clean[['Abnormal_Temperature', 'Abnormal_Vibration', 'Critical_Battery']]
    .sum(axis=1)
)

abnormal_by_factory = df_clean.groupby('Factory')['Abnormal_Sensor_Count'].sum().sort_values(ascending=False)
display(abnormal_by_factory.to_frame('Abnormal_Sensor_Readings'))

top_factory = abnormal_by_factory.idxmax()
print(f'Factory with highest number of abnormal sensor readings: {top_factory}')

### 25. Percentage of time each device operates under warning/critical conditions

The calculation uses the proportion of recorded observations for each device whose overall health is Warning or Critical.

In [ ]:
device_condition_percentage = (
    df_clean.assign(Warning_or_Critical=df_clean['Machine_Health'].isin(['Warning', 'Critical']))
    .groupby('Device')['Warning_or_Critical']
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)
display(device_condition_percentage.to_frame('Warning_Critical_Percentage'))

### 26. Device requiring the highest maintenance priority

A maintenance priority score is calculated using critical-condition frequency, warning/critical percentage, and abnormal sensor counts. The device with the highest score is selected as the highest priority.

In [ ]:
device_summary = df_clean.groupby('Device').agg(
    Critical_Count=('Machine_Health', lambda x: (x == 'Critical').sum()),
    Warning_Count=('Machine_Health', lambda x: (x == 'Warning').sum()),
    Abnormal_Sensor_Readings=('Abnormal_Sensor_Count', 'sum'),
    Total_Readings=('Machine_Health', 'size')
)

device_summary['Warning_Critical_Percentage'] = (
    (device_summary['Critical_Count'] + device_summary['Warning_Count']) /
    device_summary['Total_Readings'] * 100
).round(2)

# Transparent score: each component is normalized to its maximum.
for col in ['Critical_Count', 'Warning_Critical_Percentage', 'Abnormal_Sensor_Readings']:
    max_value = device_summary[col].max()
    device_summary[col + '_Normalized'] = (
        device_summary[col] / max_value if max_value != 0 else 0
    )

device_summary['Maintenance_Priority_Score'] = (
    0.5 * device_summary['Critical_Count_Normalized'] +
    0.3 * device_summary['Warning_Critical_Percentage_Normalized'] +
    0.2 * device_summary['Abnormal_Sensor_Readings_Normalized']
).round(4)

device_summary = device_summary.sort_values('Maintenance_Priority_Score', ascending=False)
display(device_summary)

highest_priority_device = device_summary.index[0]
print(f'Highest maintenance priority device: {highest_priority_device}')

## Summary

The notebook completes all requested data preparation, cleaning, time-series, status classification, and advanced maintenance analysis tasks. Run all cells from top to bottom to generate the dataset-specific outputs.